# PPG + ECG → Blood Pressure → Cardiovascular Risk

**Step 2 of the project: training.**

1. Model 1 (`BPNet`): PPG + ECG (+ age, sex, BMI) → SBP and DBP
2. Model 2 (`RiskNet`): PPG + ECG + BP from model 1 + patient information → No / Low / Moderate / High risk

The signal part of both models is an RNN (`ARCH = "gru"` or `"lstm"`) or the older CNN (`"cnn"`).

**How to run on Kaggle**
- Settings → Accelerator: **GPU**, Internet: **On**
- Check the settings cell (`TRY`, `DATA`, `ARCH`), then **Save Version → Save & Run All**
- A checkpoint is saved after every epoch in `/kaggle/working/<TRY>/checkpoints`. If the run stops, add the old run's output with *Add Input → Notebook Output* and run again: it continues from the last epoch.
- At the end download everything in `/kaggle/working/<TRY>/weights` and push it to `<TRY>/weights/` in the repo.

In [ ]:
import os, sys, glob, shutil, json

REPO_URL = "https://github.com/NeelakandanNC/final_year_project"

if os.path.exists("models.py"):              # running locally inside the repo
    REPO_DIR = "."
else:                                        # on Kaggle: download the repo (code + data)
    REPO_DIR = "/tmp/final_year_project"
    if not os.path.exists(REPO_DIR):
        os.system(f"git clone --depth 1 {REPO_URL} {REPO_DIR}")
sys.path.append(REPO_DIR)
print("Repo folder:", REPO_DIR)

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.ensemble import HistGradientBoostingClassifier

from models import (BPNet, RiskNet, normalize, patient_features, clinical_features,
                    RISK_NAMES, BP_MEAN, BP_STD, N_PATIENT)

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
np.random.seed(42)
print("Device:", device)

## Settings

In [ ]:
TRY = "try3"                  # folder where the results are saved (try3, try4, ...)
DATA = "try3"                 # folder with the collected data
ARCH = "gru"                  # signal model: "gru" or "lstm" (RNN), or "cnn"
ACT = "relu"                  # activation in the dense layers: "relu", "gelu" or "silu" (tested: no real difference)
BP_USE_PATIENT_INFO = True    # give age, sex, BMI to the BP model as well
RISK_INPUTS = "clinical"      # "basic" = age, sex, BMI   |   "clinical" = all pre-op info in clinical.csv
FREEZE_ENCODER = True         # risk model reuses the BP model's signal features (stops overfitting)
AUGMENT = True                # small random noise + time shift on training signals

EPOCHS_BP = 40
EPOCHS_RISK = 30
PATIENCE = 8                  # stop early if validation has not improved for this many epochs
BATCH_SIZE = 256
LR_BP = 2e-3                  # peak learning rate (OneCycle schedule)
LR_RISK = 1e-3

OUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "outputs"
CKPT_DIR = f"{OUT_DIR}/{TRY}/checkpoints"
WEIGHTS_DIR = f"{OUT_DIR}/{TRY}/weights"
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(WEIGHTS_DIR, exist_ok=True)

## Load the data collected by `collect_data.py`

In [ ]:
files = sorted(glob.glob(f"{REPO_DIR}/{DATA}/data/part_*.npz"))
assert files, f"No data found in {DATA}/data. Run collect_data.py and push the data first."
parts = [np.load(f) for f in files]
data = {k: np.concatenate([p[k] for p in parts]) for k in parts[0].files}

X = normalize(data["X"])                       # (N, 2, 1000): PPG, ECG
BP = np.stack([data["sbp"], data["dbp"]], 1)   # (N, 2): SBP, DBP in mmHg
risk = data["risk"]
subject = data["subjectid"]
caseid = data["caseid"]
PAT = patient_features(data["age"], data["sex"], data["bmi"])    # age, sex, BMI for every segment

if RISK_INPUTS == "clinical":
    clin_path = f"{REPO_DIR}/{DATA}/data/clinical.csv"
    assert os.path.exists(clin_path), "clinical.csv missing: run the new collect_data.py on this folder"
    clinical = pd.read_csv(clin_path).set_index("caseid")
    clin_seg = clinical.loc[caseid].reset_index()                  # clinical row for every segment

print("Segments:", len(X), "| Patients:", len(np.unique(caseid)))
print("SBP mean %.1f ± %.1f   DBP mean %.1f ± %.1f" % (BP[:, 0].mean(), BP[:, 0].std(), BP[:, 1].mean(), BP[:, 1].std()))
for r, name in enumerate(RISK_NAMES):
    print(f"{name:14s}: {len(np.unique(caseid[risk == r])):5d} patients, {np.sum(risk == r):6d} segments")

In [ ]:
# one example segment
i = 0
t = np.arange(1000) / 100
fig, ax = plt.subplots(2, 1, figsize=(12, 4), sharex=True)
ax[0].plot(t, data["X"][i, 0]); ax[0].set_ylabel("PPG")
ax[1].plot(t, data["X"][i, 1], color="tab:red"); ax[1].set_ylabel("ECG")
ax[1].set_xlabel("time (s)")
ax[0].set_title(f"SBP {BP[i, 0]:.0f} / DBP {BP[i, 1]:.0f} mmHg   |   {RISK_NAMES[risk[i]]}")
plt.show()

## Train / validation / test split (by patient, so no patient is in two sets)

In [ ]:
subj_risk = pd.Series(risk, index=subject).groupby(level=0).first()
rng = np.random.default_rng(42)
test_subj, val_subj = [], []
for r in range(4):                                  # split each risk class 70 / 15 / 15
    s = subj_risk[subj_risk == r].index.values.copy()
    rng.shuffle(s)
    n = len(s)
    test_subj += list(s[:int(0.15 * n)])
    val_subj += list(s[int(0.15 * n):int(0.30 * n)])

split = np.where(np.isin(subject, test_subj), "test", np.where(np.isin(subject, val_subj), "val", "train"))

# inputs for the risk model (scaling is learned from the training patients only)
if RISK_INPUTS == "clinical":
    CLIN, clin_stats = clinical_features(clin_seg[split == "train"].drop_duplicates("caseid"))
    CLIN, _ = clinical_features(clin_seg, clin_stats)
else:
    CLIN, clin_stats = PAT, None
print("Risk model patient inputs:", CLIN.shape[1], "values per patient")

def make_loader(name, shuffle):
    m = split == name
    ds = TensorDataset(torch.tensor(X[m]), torch.tensor(BP[m], dtype=torch.float32),
                       torch.tensor(risk[m]), torch.tensor(PAT[m]), torch.tensor(CLIN[m]))
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)

train_dl, val_dl, test_dl = make_loader("train", True), make_loader("val", False), make_loader("test", False)
for name in ["train", "val", "test"]:
    print(f"{name:5s}: {len(np.unique(caseid[split == name])):5d} patients, {np.sum(split == name):6d} segments")

## Training loop (AdamW + OneCycle learning rate, checkpoint after every epoch)

In [ ]:
def augment(x):
    # small random noise and a random time shift (up to 0.5 s), only while training
    x = x + 0.05 * torch.randn_like(x)
    return torch.roll(x, shifts=int(np.random.randint(-50, 51)), dims=2)


def find_checkpoint(name):
    path = f"{CKPT_DIR}/{name}_last.pt"
    if os.path.exists(path):
        return path
    old = glob.glob(f"/kaggle/input/**/{TRY}/checkpoints/{name}_last.pt", recursive=True)   # older run
    if old:
        for f in [old[0], old[0].replace("_last.pt", "_best.pt")]:
            if os.path.exists(f):
                shutil.copy(f, CKPT_DIR)
        return f"{CKPT_DIR}/{name}_last.pt"
    return None


def fit(name, model, batch_loss, epochs, lr, val_score=None):
    # val_score(model) -> number, higher is better. Without it the validation loss is used.
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * len(train_dl))
    start, best, history = 0, float("inf"), {"train": [], "val": []}

    ckpt = find_checkpoint(name)
    if ckpt:
        state = torch.load(ckpt, map_location=device, weights_only=False)
        model.load_state_dict(state["model"])
        opt.load_state_dict(state["optimizer"])
        sched.load_state_dict(state["scheduler"])
        start, best, history = state["epoch"] + 1, state["best"], state["history"]
        print(f"Resuming {name} from epoch {start + 1}")

    for epoch in range(start, epochs):
        if history["val"] and len(history["val"]) - 1 - int(np.argmin(history["val"])) >= PATIENCE:
            print(f"No improvement for {PATIENCE} epochs, stopping early")
            break
        model.train()
        train_losses = []
        for batch in train_dl:
            loss = batch_loss(model, batch)
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(params, 1.0)        # keeps RNN training stable
            opt.step()
            sched.step()
            train_losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            val_loss = np.mean([batch_loss(model, b).item() for b in val_dl])
            val = -val_score(model) if val_score else val_loss          # lower is better
        train_loss = np.mean(train_losses)
        history["train"].append(train_loss)
        history["val"].append(val)

        msg = f"  val score {-val:.3f}" if val_score else ""
        if val < best:
            best = val
            torch.save(model.state_dict(), f"{CKPT_DIR}/{name}_best.pt")
            msg += "  <- best"
        torch.save({"epoch": epoch, "model": model.state_dict(), "optimizer": opt.state_dict(),
                    "scheduler": sched.state_dict(), "best": best, "history": history},
                   f"{CKPT_DIR}/{name}_tmp.pt")
        os.replace(f"{CKPT_DIR}/{name}_tmp.pt", f"{CKPT_DIR}/{name}_last.pt")
        print(f"{name}  epoch {epoch + 1}/{epochs}  train loss {train_loss:.4f}  val loss {val_loss:.4f}{msg}")

    model.load_state_dict(torch.load(f"{CKPT_DIR}/{name}_best.pt", map_location=device))
    plt.figure(figsize=(6, 3))
    plt.plot(history["train"], label="train loss")
    plt.plot(history["val"], label="val score (negative)" if val_score else "val loss")
    plt.title(name); plt.xlabel("epoch"); plt.legend(); plt.show()
    return model

## Model 1: BP estimation (PPG + ECG → SBP, DBP)

In [ ]:
bp_mean, bp_std = BP_MEAN.to(device), BP_STD.to(device)

def bp_loss(model, batch):
    x, bp, pat = batch[0].to(device), batch[1].to(device), batch[3].to(device)
    if model.training and AUGMENT:
        x = augment(x)
    return nn.functional.smooth_l1_loss(model(x, pat), (bp - bp_mean) / bp_std)

bp_model = BPNet(arch=ARCH, act=ACT, n_extra=N_PATIENT if BP_USE_PATIENT_INFO else 0).to(device)
print(f"BPNet ({ARCH}): {sum(p.numel() for p in bp_model.parameters()) / 1e3:.0f}k parameters")
bp_model = fit("bp_model", bp_model, bp_loss, EPOCHS_BP, LR_BP)

In [ ]:
def predict_bp(loader):
    bp_model.eval()
    out = []
    with torch.no_grad():
        for batch in loader:
            out.append(bp_model.predict_bp(batch[0].to(device), batch[3].to(device)).cpu().numpy())
    return np.concatenate(out)

pred = predict_bp(test_dl)
true = BP[split == "test"]

def bp_lines(p, t):
    lines = []
    for i, name in enumerate(["SBP", "DBP"]):
        err = p[:, i] - t[:, i]
        within = [np.mean(np.abs(err) <= k) * 100 for k in (5, 10, 15)]
        lines.append(f"  {name}: MAE {np.abs(err).mean():.2f} mmHg | ME {err.mean():.2f} ± SD {err.std():.2f} mmHg | "
                     f"within 5/10/15 mmHg: {within[0]:.0f}% / {within[1]:.0f}% / {within[2]:.0f}%")
    return lines

# baseline: always guess the average BP of the training set (the model must beat this)
guess = np.tile(BP[split == "train"].mean(0), (len(true), 1))

report = [f"{TRY} results  (data: {DATA}, model: {ARCH}, activation: {ACT})", "",
          "BP model (test set, new patients)"]
report += ["Baseline (always guess the average BP):"] + bp_lines(guess, true)
report += [f"{ARCH.upper()} model:"] + bp_lines(pred, true)
report.append("(AAMI standard: ME within ±5 mmHg and SD below 8 mmHg)")
print("\n".join(report))

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for i, name in enumerate(["SBP", "DBP"]):
    ax[i].scatter(true[:, i], pred[:, i], s=2, alpha=0.3)
    lo, hi = true[:, i].min(), true[:, i].max()
    ax[i].plot([lo, hi], [lo, hi], "r--")
    ax[i].set_xlabel(f"true {name} (mmHg)"); ax[i].set_ylabel(f"predicted {name} (mmHg)"); ax[i].set_title(name)
plt.tight_layout(); plt.show()

## Model 2: Cardiovascular risk (PPG + ECG + predicted BP + patient info → 4 classes)

In [ ]:
# freeze the BP model, its predictions are used as an input
bp_model.eval()
for p in bp_model.parameters():
    p.requires_grad = False

# class weights because the risk classes are not balanced
counts = np.bincount(risk[split == "train"], minlength=4)
class_w = torch.tensor(counts.sum() / (4 * np.maximum(counts, 1)), dtype=torch.float32, device=device)
ce = nn.CrossEntropyLoss(weight=class_w, label_smoothing=0.1)

def risk_loss(model, batch):
    x, _, y, pat, clin = [t.to(device) for t in batch]
    if FREEZE_ENCODER:
        model.encoder.eval()                       # frozen part stays in evaluation mode
    with torch.no_grad():
        bp_pred = bp_model.predict_bp(x, pat)
    if model.training and AUGMENT:
        x = augment(x)
    return ce(model(x, bp_pred, clin), y)

def risk_probs(model, loader):
    model.eval()
    out = []
    with torch.no_grad():
        for batch in loader:
            x, _, _, pat, clin = [t.to(device) for t in batch]
            out.append(torch.softmax(model(x, bp_model.predict_bp(x, pat), clin), 1).cpu().numpy())
    return np.concatenate(out)

def patient_level(probs, name):
    # average the predictions of all segments of a patient
    df = pd.DataFrame(probs)
    df["caseid"] = caseid[split == name]
    df["true"] = risk[split == name]
    pat = df.groupby("caseid").mean()
    return pat["true"].astype(int).values, pat[[0, 1, 2, 3]].values.argmax(1)

def val_patient_accuracy(model):
    t, p = patient_level(risk_probs(model, val_dl), "val")
    return accuracy_score(t, p)

risk_model = RiskNet(n_extra=CLIN.shape[1], arch=ARCH, act=ACT).to(device)
risk_model.encoder.load_state_dict(bp_model.encoder.state_dict())   # start from the BP model's signal features
if FREEZE_ENCODER:
    for p in risk_model.encoder.parameters():
        p.requires_grad = False
risk_model = fit("risk_model", risk_model, risk_loss, EPOCHS_RISK, LR_RISK, val_score=val_patient_accuracy)

In [ ]:
probs = risk_probs(risk_model, test_dl)
y_true = risk[split == "test"]
y_pred = probs.argmax(1)
p_true, p_pred = patient_level(probs, "test")

# comparison: patient information only, no signals (gradient boosting on one row per patient)
first = pd.Series(np.arange(len(caseid))).groupby(caseid).first().values
tr_idx = first[split[first] == "train"]
te_idx = first[split[first] == "test"]
gb = HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=300, class_weight="balanced", random_state=0)
gb.fit(CLIN[tr_idx], risk[tr_idx])
gb_pred = gb.predict(CLIN[te_idx])

def risk_lines(t, p):
    within1 = np.mean(np.abs(t - p) <= 1) * 100
    recall = [np.mean(p[t == r] == r) * 100 if np.any(t == r) else float("nan") for r in range(4)]
    return (f"accuracy {accuracy_score(t, p) * 100:.1f}% | macro F1 {f1_score(t, p, average='macro'):.3f} | "
            f"within one class {within1:.1f}% | per class (No/Low/Mod/High) "
            + " / ".join(f"{r:.0f}%" for r in recall))

report.append("")
report.append(f"Risk model (test set), inputs: PPG + ECG + predicted BP + "
              + ("all pre-op clinical info" if RISK_INPUTS == "clinical" else "age, sex, BMI"))
report.append(f"  Segment level : {risk_lines(y_true, y_pred)}")
report.append(f"  Patient level : {risk_lines(p_true, p_pred)}")
report.append(f"  Patient info only, no signals (gradient boosting): {risk_lines(risk[te_idx], gb_pred)}")
report.append("(random guessing = 25% accuracy; 'within one class' counts Low vs Moderate etc. as close)")
print("\n".join(report[-5:]))

fig, ax = plt.subplots(1, 3, figsize=(18, 5))
for a, (t, p, title) in zip(ax, [(y_true, y_pred, "Segments"), (p_true, p_pred, "Patients"),
                                 (risk[te_idx], gb_pred, "Patient info only")]):
    cm = confusion_matrix(t, p, labels=range(4))
    ConfusionMatrixDisplay(cm, display_labels=RISK_NAMES).plot(ax=a, colorbar=False, xticks_rotation=30)
    a.set_title(title)
plt.tight_layout(); plt.show()

## Save the final weights

In [ ]:
torch.save(bp_model.state_dict(), f"{WEIGHTS_DIR}/bp_model.pt")
torch.save(risk_model.state_dict(), f"{WEIGHTS_DIR}/risk_model.pt")
config = {"data": DATA, "arch": ARCH, "act": ACT,
          "bp_n_extra": N_PATIENT if BP_USE_PATIENT_INFO else 0,
          "risk_inputs": RISK_INPUTS, "risk_n_extra": int(CLIN.shape[1]), "clinical_stats": clin_stats}
with open(f"{WEIGHTS_DIR}/config.json", "w") as f:
    json.dump(config, f, indent=1)
with open(f"{WEIGHTS_DIR}/results.txt", "w") as f:
    f.write("\n".join(report) + "\n")

for f in sorted(os.listdir(WEIGHTS_DIR)):
    print(f, f"{os.path.getsize(f'{WEIGHTS_DIR}/{f}') / 1e6:.2f} MB")
print(f"\nDownload these files and put them in the {TRY}/weights/ folder of the GitHub repo.")